
# 3. Desarrollo del análisis

## A. Planteamiento de la comparación

En esta actividad se realizará una comparación de las condiciones climáticas de Medellín, Colombia, y Quito, Ecuador, con el propósito de identificar similitudes y diferencias en sus variables de temperatura y precipitación durante un mismo periodo.

### Predicción o expectativa

Se espera que Medellín y Quito presenten algunas similitudes en sus condiciones climáticas debido a su ubicación geográfica cercana a la línea ecuatorial. Sin embargo, se considera que pueden existir diferencias en sus temperaturas medias, temperaturas máximas y patrones de precipitación.

Como expectativa inicial, se plantea que las dos ciudades tendrán comportamientos distintos en algunos meses del año, tanto en sus niveles de temperatura como en la cantidad de lluvia registrada. Esta predicción será contrastada posteriormente mediante el análisis de los datos y las gráficas.

### Periodo de estudio

Para garantizar una comparación válida, se utilizará el periodo comprendido entre el **1 de enero y el 31 de diciembre de 2024** en ambas ciudades.

Se trabajará con una frecuencia diaria, lo que permitirá comparar los registros correspondientes a las mismas fechas.

### Variables seleccionadas

Las variables climáticas que se utilizarán son:

| Variable | Descripción | Unidad |
|---|---|---|
| temperature_2m_mean | Temperatura media diaria a 2 metros de altura | °C |
| temperature_2m_max | Temperatura máxima diaria a 2 metros de altura | °C |
| precipitation_sum | Precipitación acumulada diaria | mm |

Estas variables permitirán analizar el comportamiento de las temperaturas y las precipitaciones, identificar patrones y establecer diferencias entre las dos ciudades.



## B. Carga y revisión de los datos

Se cargarán los archivos climáticos de Medellín y Quito mediante Pandas. Posteriormente, se verificará su estructura, tipos de datos, fechas, valores faltantes y registros duplicados para garantizar la calidad de la información.



### B.1. Importación de librerías y descarga de datos

Se importan las librerías necesarias y se consultan los datos históricos de Open-Meteo para ambas ciudades, utilizando el modelo ERA5-Land y el periodo establecido.



In [8]:

import requests
import pandas as pd
from pathlib import Path

# Carpeta donde se guardarán los archivos
carpeta_datos = Path("datos")
carpeta_datos.mkdir(exist_ok=True)

# Configuración de las ciudades
ciudades = {
    "medellin": {
        "latitude": 6.2442,
        "longitude": -75.5812,
        "timezone": "America/Bogota"
    },
    "quito": {
        "latitude": -0.1807,
        "longitude": -78.4678,
        "timezone": "America/Guayaquil"
    }
}

url = "https://archive-api.open-meteo.com/v1/archive"

for ciudad, ubicacion in ciudades.items():

    parametros = {
        **ubicacion,
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "precipitation_sum"
        ],
        "models": "best_match"
    }

    respuesta = requests.get(url, params=parametros, timeout=60)
    respuesta.raise_for_status()

    datos = respuesta.json()

    df = pd.DataFrame(datos["daily"])
    df.rename(columns={"time": "fecha"}, inplace=True)

    ruta = carpeta_datos / f"{ciudad}.csv"
    df.to_csv(ruta, index=False)

    print(f"{ciudad}: {len(df)} registros descargados.")
    print(f"Archivo guardado: {ruta}")


medellin: 366 registros descargados.
Archivo guardado: datos\medellin.csv
quito: 366 registros descargados.
Archivo guardado: datos\quito.csv



### B.2. Carga de los archivos CSV

Se leen los archivos descargados mediante Pandas y se almacenan en DataFrames independientes para su posterior exploración.


In [9]:

from pathlib import Path

import pandas as pd
import numpy as np

# Definir la ubicación de los archivos
carpeta_datos = Path("datos")

ruta_medellin = carpeta_datos / "medellin.csv"
ruta_quito = carpeta_datos / "quito.csv"

# Verificar que los archivos existan
for ruta in [ruta_medellin, ruta_quito]:
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}"
        )

# Cargar los conjuntos de datos
df_medellin = pd.read_csv(ruta_medellin)
df_quito = pd.read_csv(ruta_quito)

print("Archivos cargados correctamente.")
print(f"Medellín: {len(df_medellin)} registros")
print(f"Quito: {len(df_quito)} registros")


Archivos cargados correctamente.
Medellín: 366 registros
Quito: 366 registros


### B.3. Inspección de los conjuntos de datos

In [10]:
print("ESTRUCTURA DE MEDELLÍN")
display(df_medellin.head())

print("\nESTRUCTURA DE QUITO")
display(df_quito.head())

ESTRUCTURA DE MEDELLÍN


,fecha,temperature_2m_mean,temperature_2m_max,precipitation_sum
0,2024-01-01,20.1,26.5,1.5
1,2024-01-02,21.1,27.5,0.5
2,2024-01-03,20.8,26.8,0.0
3,2024-01-04,20.4,27.1,0.3
4,2024-01-05,20.2,27.0,0.4



ESTRUCTURA DE QUITO


,fecha,temperature_2m_mean,temperature_2m_max,precipitation_sum
0,2024-01-01,11.9,17.0,27.3
1,2024-01-02,12.0,17.2,10.1
2,2024-01-03,12.0,15.5,22.3
3,2024-01-04,11.7,15.3,33.8
4,2024-01-05,12.4,17.0,2.3


In [11]:
print("VALORES FALTANTES")
print("\nMedellín:")
print(df_medellin.isnull().sum())

print("\nQuito:")
print(df_quito.isnull().sum())

VALORES FALTANTES

Medellín:
fecha                  0
temperature_2m_mean    0
temperature_2m_max     0
precipitation_sum      0
dtype: int64

Quito:
fecha                  0
temperature_2m_mean    0
temperature_2m_max     0
precipitation_sum      0
dtype: int64


### B.4. Verificación de fechas y registros duplicados

In [12]:
print("REGISTROS DUPLICADOS")

print("Medellín:", df_medellin["fecha"].duplicated().sum())
print("Quito:", df_quito["fecha"].duplicated().sum())

print("\nPERIODO DE LOS DATOS")

print("Medellín:", df_medellin["fecha"].min(), "a", df_medellin["fecha"].max())
print("Quito:", df_quito["fecha"].min(), "a", df_quito["fecha"].max())

REGISTROS DUPLICADOS
Medellín: 0
Quito: 0

PERIODO DE LOS DATOS
Medellín: 2024-01-01 a 2024-12-31
Quito: 2024-01-01 a 2024-12-31


### B.5. Conversión de tipos y fechas comunes

In [13]:
# Convertir fechas
df_medellin["fecha"] = pd.to_datetime(df_medellin["fecha"])
df_quito["fecha"] = pd.to_datetime(df_quito["fecha"])

# Convertir mediciones a valores numéricos
columnas = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "precipitation_sum"
]

for columna in columnas:
    df_medellin[columna] = pd.to_numeric(df_medellin[columna], errors="coerce")
    df_quito[columna] = pd.to_numeric(df_quito[columna], errors="coerce")

# Calcular dimensiones y fechas válidas en común
print("DIMENSIONES")
print("Medellín:", df_medellin.shape)
print("Quito:", df_quito.shape)

fechas_comunes = set(df_medellin["fecha"].dropna()) & set(df_quito["fecha"].dropna())

print("\nOBSERVACIONES DISPONIBLES PARA COMPARACIÓN:")
print(len(fechas_comunes))

DIMENSIONES
Medellín: (366, 4)
Quito: (366, 4)

OBSERVACIONES DISPONIBLES PARA COMPARACIÓN:
366


## C. Analiza con NumPy

### C.1. Construcción de arrays NumPy

In [14]:
# Crear arrays de temperatura y precipitación
temperatura_media_medellin = df_medellin["temperature_2m_mean"].to_numpy()
temperatura_max_medellin = df_medellin["temperature_2m_max"].to_numpy()
precipitacion_medellin = df_medellin["precipitation_sum"].to_numpy()

temperatura_media_quito = df_quito["temperature_2m_mean"].to_numpy()
temperatura_max_quito = df_quito["temperature_2m_max"].to_numpy()
precipitacion_quito = df_quito["precipitation_sum"].to_numpy()

# Verificar los arrays
print("Arrays de Medellín:", temperatura_media_medellin.shape)
print("Arrays de Quito:", temperatura_media_quito.shape)

Arrays de Medellín: (366,)
Arrays de Quito: (366,)


### C.2. Inspección mediante índices y cortes

In [15]:
# Inspeccionar los primeros 7 días de enero de 2024
print("MEDELLÍN - PRIMEROS 7 DÍAS")
print("Temperatura media:", temperatura_media_medellin[:7])
print("Precipitación:", precipitacion_medellin[:7])

print("\nQUITO - PRIMEROS 7 DÍAS")
print("Temperatura media:", temperatura_media_quito[:7])
print("Precipitación:", precipitacion_quito[:7])

MEDELLÍN - PRIMEROS 7 DÍAS
Temperatura media: [20.1 21.1 20.8 20.4 20.2 20.3 20.9]
Precipitación: [1.5 0.5 0.  0.3 0.4 0.  0. ]

QUITO - PRIMEROS 7 DÍAS
Temperatura media: [11.9 12.  12.  11.7 12.4 12.4 12.5]
Precipitación: [27.3 10.1 22.3 33.8  2.3  4.9  0. ]


### C.3. Selección de días mediante máscaras booleanas

Se crearán máscaras booleanas con NumPy para identificar los días lluviosos (precipitación superior a 0 mm) y los días cuya temperatura media supere los 20 °C. Se aplicará el mismo umbral de temperatura a ambas ciudades para facilitar la comparación.

In [16]:
# Definir el umbral de temperatura
umbral_temperatura = 20

# Crear máscaras para días lluviosos
lluvia_medellin = precipitacion_medellin > 0
lluvia_quito = precipitacion_quito > 0

# Crear máscaras para días con temperatura superior al umbral
calor_medellin = temperatura_media_medellin > umbral_temperatura
calor_quito = temperatura_media_quito > umbral_temperatura

# Contar los días seleccionados
print("DÍAS LLUVIOSOS")
print("Medellín:", np.sum(lluvia_medellin))
print("Quito:", np.sum(lluvia_quito))

print("\nDÍAS CON TEMPERATURA MEDIA SUPERIOR A", umbral_temperatura, "°C")
print("Medellín:", np.sum(calor_medellin))
print("Quito:", np.sum(calor_quito))

DÍAS LLUVIOSOS
Medellín: 336
Quito: 288

DÍAS CON TEMPERATURA MEDIA SUPERIOR A 20 °C
Medellín: 297
Quito: 0


### C.4. Cálculo de medidas descriptivas

Se calcularán mediante NumPy las medidas estadísticas de cada ciudad: temperatura media promedio, mínima y máxima, precipitación total y número de días lluviosos. Estos resultados permitirán comparar el comportamiento climático de Medellín y Quito durante 2024.

In [17]:
# Medidas descriptivas de Medellín
print("MEDELLÍN")
print("Temperatura media promedio:", np.mean(temperatura_media_medellin))
print("Temperatura mínima:", np.min(temperatura_media_medellin))
print("Temperatura máxima:", np.max(temperatura_max_medellin))
print("Precipitación total:", np.sum(precipitacion_medellin))
print("Días lluviosos:", np.sum(lluvia_medellin))

# Medidas descriptivas de Quito
print("\nQUITO")
print("Temperatura media promedio:", np.mean(temperatura_media_quito))
print("Temperatura mínima:", np.min(temperatura_media_quito))
print("Temperatura máxima:", np.max(temperatura_max_quito))
print("Precipitación total:", np.sum(precipitacion_quito))
print("Días lluviosos:", np.sum(lluvia_quito))

MEDELLÍN
Temperatura media promedio: 20.91393442622951
Temperatura mínima: 18.5
Temperatura máxima: 31.9
Precipitación total: 2176.3
Días lluviosos: 336

QUITO
Temperatura media promedio: 12.493989071038254
Temperatura mínima: 10.7
Temperatura máxima: 22.9
Precipitación total: 3044.5999999999995
Días lluviosos: 288


### C.5. Comparación de temperaturas entre ciudades

Se calculará la diferencia entre las temperaturas medias promedio de Medellín y Quito utilizando NumPy. Esto permitirá identificar cuál ciudad presentó una temperatura promedio más alta durante 2024 y cuantificar la diferencia en grados Celsius.

In [18]:
# Calcular temperaturas medias de ambas ciudades
promedio_medellin = np.mean(temperatura_media_medellin)
promedio_quito = np.mean(temperatura_media_quito)

# Calcular diferencia de temperaturas
diferencia_temperatura = np.abs(promedio_medellin - promedio_quito)

print("COMPARACIÓN DE TEMPERATURAS")
print(f"Medellín: {promedio_medellin:.2f} °C")
print(f"Quito: {promedio_quito:.2f} °C")
print(f"Diferencia: {diferencia_temperatura:.2f} °C")

COMPARACIÓN DE TEMPERATURAS
Medellín: 20.91 °C
Quito: 12.49 °C
Diferencia: 8.42 °C


### C.6. Interpretación de resultados

Durante 2024, Medellín presentó una temperatura media promedio de 20,91 °C, mientras que Quito registró 12,49 °C, lo que representa una diferencia de 8,42 °C. 

En cuanto a la precipitación, Quito acumuló 3044,6 mm, superando los 2176,3 mm de Medellín. Sin embargo, Medellín presentó más días lluviosos (336) que Quito (288).

Los resultados muestran que Medellín tuvo condiciones más cálidas, mientras que Quito registró una mayor cantidad de precipitación acumulada durante el periodo analizado.